# Pesquisa Avançada: Detecção de Câncer via Feature Engineering Progressiva
**Objetivo:** Medir o ganho incremental de Cor, Textura e Espectro na classificação de lesões de pele.

---

In [ ]:
!pip install datasets tensorflow pandas matplotlib seaborn scikit-learn scikit-image imbalanced-learn -q
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import cv2
import gc
import tensorflow as tf
from datasets import load_dataset
from skimage.feature import graycomatrix, graycoprops, local_binary_pattern
from skimage.color import rgb2lab, rgb2hsv
from skimage.measure import moments, moments_hu
from skimage.filters import threshold_otsu
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.decomposition import PCA
from sklearn.feature_selection import VarianceThreshold
from sklearn.manifold import TSNE
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight
from imblearn.over_sampling import RandomOverSampler
from tensorflow.keras import layers, models, optimizers, callbacks

print("--- Configurando Acelerador ---")
try:
    tpu = tf.distribute.cluster_resolver.TPUClusterResolver(tpu='local')
    tf.config.experimental_connect_to_cluster(tpu)
    tf.tpu.experimental.initialize_tpu_system(tpu)
    strategy = tf.distribute.TPUStrategy(tpu)
    print('Hardware: TPU Ativa!')
except:
    gpus = tf.config.list_physical_devices('GPU')
    if len(gpus) > 1:
        strategy = tf.distribute.MirroredStrategy()
        print(f'Hardware: {len(gpus)} GPUs Ativas')
    elif len(gpus) == 1:
        strategy = tf.distribute.OneDeviceStrategy(device="/gpu:0")
        print('Hardware: 1 GPU Ativa')
    else:
        strategy = tf.distribute.get_strategy()
        print('Hardware: CPU')

IMG_SIZE = (224, 224)
BATCH_SIZE = 16 * strategy.num_replicas_in_sync

## 1. Carregamento e Balanceamento (Máx 4k Nevi)
Usamos streaming para garantir que o download não trave no Kaggle.

In [ ]:
print("--- Iniciando Streaming Híbrido ---")
try:
    from kaggle_secrets import UserSecretsClient
    hf_token = UserSecretsClient().get_secret("HF_TOKEN")
except: hf_token = None

ds_stream = load_dataset("marmal88/skin_cancer", split='train', streaming=True, token=hf_token)

limit_nevi = 4000
counts = {}
X_raw, y_labels = [], []

for example in ds_stream:
    lbl = example['dx']
    counts[lbl] = counts.get(lbl, 0)
    
    if lbl == 'melanocytic_Nevi' and counts[lbl] >= limit_nevi:
        continue
        
    img = example['image'].convert('RGB').resize(IMG_SIZE)
    X_raw.append(np.array(img, dtype='uint8'))
    y_labels.append(lbl)
    counts[lbl] += 1
    if sum(counts.values()) % 1000 == 0: print(f"Capturadas: {sum(counts.values())} | {counts}")

X_raw = np.array(X_raw)
y_labels = np.array(y_labels)
gc.collect()
print(f"Dataset Finalizado: {X_raw.shape}")

## 2. Feature Engineering Progressiva
Criamos extratores para Cor, Textura e Espectro.

In [ ]:
print("--- Extração Avançada: DullRazor + ABCD (GLCM+LBP) ---")

def dull_razor(img_uint8):
    gray = cv2.cvtColor(img_uint8, cv2.COLOR_RGB2GRAY)
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (9, 9))
    blackhat = cv2.morphologyEx(gray, cv2.MORPH_BLACKHAT, kernel)
    _, mask = cv2.threshold(blackhat, 10, 255, cv2.THRESH_BINARY)
    dst = cv2.inpaint(img_uint8, mask, 1, cv2.INPAINT_TELEA)
    return dst

def extract_all_features(img_raw):
    img_uint8 = dull_razor(img_raw)
    img_f = img_uint8.astype('float32') / 255.0
    img_gray = cv2.cvtColor(img_uint8, cv2.COLOR_RGB2GRAY)
    
    # 1. COR (RGB + HSV)
    color_rgb = img_f.mean(axis=(0,1))
    img_hsv = rgb2hsv(img_f)
    color_hsv = img_hsv.mean(axis=(0,1))
    color_feat = np.concatenate([color_rgb, color_hsv])
    
    # 2. TEXTURA (GLCM + LBP)
    glcm = graycomatrix(img_gray, [1], [0], levels=256, symmetric=True, normed=True)
    texture_glcm = [
        graycoprops(glcm, 'contrast')[0,0], 
        graycoprops(glcm, 'homogeneity')[0,0],
        graycoprops(glcm, 'energy')[0,0],
        graycoprops(glcm, 'correlation')[0,0]
    ]
    lbp = local_binary_pattern(img_gray, 8, 1, method='uniform')
    (hist, _) = np.histogram(lbp.ravel(), bins=np.arange(0, 13), range=(0, 12))
    texture_lbp = hist.astype("float")
    texture_lbp /= (texture_lbp.sum() + 1e-7)
    texture_feat = np.concatenate([texture_glcm, texture_lbp])
    
    # 3. FORMA (Hu Moments)
    try:
        thresh = threshold_otsu(img_gray)
        binary = (img_gray > thresh).astype(np.uint8)
        if binary[0,0] == 1: binary = 1 - binary
        mu = moments(binary)
        hu = moments_hu(mu)
        hu = -np.sign(hu) * np.log10(np.abs(hu) + 1e-12)
    except:
        hu = np.zeros(7)
    shape_feat = hu
    
    return color_feat, texture_feat, shape_feat

print("Processando extração ABCD...")
feats = [extract_all_features(X_raw[i]) for i in range(len(X_raw))]
f_color = np.nan_to_num(np.array([f[0] for f in feats]))
f_texture = np.nan_to_num(np.array([f[1] for f in feats]))
f_shape = np.nan_to_num(np.array([f[2] for f in feats]))

X_final = np.hstack([f_color, f_texture, f_shape])
print(f"Finalizado! Atributos totais: {X_final.shape[1]}")

## 3. Análise Diagnóstica (EDA & PCA Visual)
Antes do treino, analisamos a separabilidade das classes e a redundância dos atributos.

In [ ]:
print("--- Gerando Insights Diagnósticos Avançados ---")
le = LabelEncoder()
y_enc = le.fit_transform(y_labels)

selector = VarianceThreshold() 
X_clean = selector.fit_transform(X_final)

# Mapa de Correlação
plt.figure(figsize=(12, 10))
sns.heatmap(pd.DataFrame(X_clean).corr(), cmap='RdBu_r', center=0)
plt.title("Matriz de Correlação (Atributos Filtrados)")
plt.show()

# t-SNE
print("Calculando t-SNE...")
tsne = TSNE(n_components=2, perplexity=30, random_state=42)
X_tsne = tsne.fit_transform(StandardScaler().fit_transform(X_clean))

plt.figure(figsize=(10, 7))
scatter = plt.scatter(X_tsne[:, 0], X_tsne[:, 1], c=y_enc, cmap='viridis', alpha=0.6)
plt.colorbar(scatter, ticks=range(len(le.classes_)), label='Classes')
plt.title("Visualização t-SNE do Espaço de Atributos")
plt.show()

X_final = X_clean

## 4. Gráfico de Ganho (Random Forest)
Medindo a importância de cada técnica.

In [ ]:
results = []
def test_rf(X_data, name):
    X_train, X_test, y_train, y_test = train_test_split(X_data, y_enc, test_size=0.2, random_state=42, stratify=y_enc)
    ros = RandomOverSampler(random_state=42)
    X_resampled, y_resampled = ros.fit_resample(X_train, y_train)
    rf = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42)
    rf.fit(X_resampled, y_resampled)
    acc = accuracy_score(y_test, rf.predict(X_test))
    results.append({'Tecnica': name, 'Acuracia': acc})
    return rf

print("Treinando modelos incrementais...")
test_rf(f_color, "A: Cor (HSV)")
test_rf(np.hstack([f_color, f_texture]), "B: Cor + Tex (GLCM+LBP)")
rf_final = test_rf(X_final, "C: Final (ABCD + DullRazor)")

sns.lineplot(data=pd.DataFrame(results), x='Tecnica', y='Acuracia', marker='o')
plt.title("Gráfico de Ganho por Técnica")
plt.show()

## 5. Deep Learning: Viabilidade de Alta Fidelidade
Modelo Dual-Input para validar o ganho com redes neurais.

In [ ]:
def categorical_focal_loss(gamma=2.0, alpha=0.25):
    """
    Implementa Focal Loss para punir erros em classes minoritárias.
    FL(pt) = -(1-pt)^gamma * log(pt)
    """
    def focal_loss(y_true, y_pred):
        # Squeeze e cast para evitar mismatch de shape em distributed training
        y_true = tf.reshape(tf.cast(y_true, tf.int32), [-1])
        y_true = tf.one_hot(y_true, depth=len(le.classes_))
        
        epsilon = tf.keras.backend.epsilon()
        y_pred = tf.clip_by_value(y_pred, epsilon, 1.0 - epsilon)
        
        # Cross Entropy com o fator de modulação Focal
        cross_entropy = -y_true * tf.math.log(y_pred)
        loss = alpha * tf.math.pow(1.0 - y_pred, gamma) * cross_entropy
        return tf.reduce_sum(loss, axis=-1)
    return focal_loss

def dl_pipeline(img_uint8, feats, label):
    img_f = tf.cast(img_uint8, tf.float32) / 255.0
    return {"rgb_input": img_f, "abcd_input": feats}, label

tf.config.optimizer.set_experimental_options({"layout_optimizer": False})
scaler_dl = StandardScaler()
X_final_scaled = scaler_dl.fit_transform(X_final)

X_tr, X_ts, y_tr, y_ts, f_tr, f_ts = train_test_split(X_raw, y_enc, X_final_scaled, test_size=0.15, stratify=y_enc)

train_ds = tf.data.Dataset.from_tensor_slices((X_tr, f_tr, y_tr)).shuffle(1000).map(dl_pipeline).batch(BATCH_SIZE).prefetch(2)
test_ds = tf.data.Dataset.from_tensor_slices((X_ts, f_ts, y_ts)).map(dl_pipeline).batch(BATCH_SIZE).prefetch(2)

# Pesos de classe agressivos
weights = compute_class_weight('balanced', classes=np.unique(y_tr), y=y_tr)
class_weight_dict = dict(enumerate(weights))
class_weight_dict[5] *= 1.5 # Reforço extra para Melanoma

with strategy.scope():
    data_aug = tf.keras.Sequential([
        layers.RandomFlip("horizontal_and_vertical"),
        layers.RandomRotation(0.2),
        layers.RandomZoom(0.1),
    ])

    # Ramo 1: Visão
    in_rgb = layers.Input(shape=(224, 224, 3), name="rgb_input")
    x1 = data_aug(in_rgb)
    x1 = tf.keras.applications.EfficientNetV2B0(include_top=False, weights='imagenet')(x1)
    x1 = layers.GlobalAveragePooling2D()(x1)
    x1 = layers.Dense(128, activation='relu')(x1)
    
    # Ramo 2: Atributos Clínicos
    in_abcd = layers.Input(shape=(X_final.shape[1],), name="abcd_input")
    x2 = layers.Dense(32, activation='relu')(in_abcd)
    
    comb = layers.Concatenate()([x1, x2])
    comb = layers.Dropout(0.4)(comb)
    out = layers.Dense(len(le.classes_), activation='softmax')(comb)
    
    model = models.Model(inputs=[in_rgb, in_abcd], outputs=out)
    model.compile(
        optimizer=optimizers.Adam(learning_rate=1e-3), 
        loss=categorical_focal_loss(gamma=2.0), 
        metrics=['accuracy']
    )

# Estratégia de Learning Rate Dinâmico
callbacks_list = [
    callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=3, min_lr=1e-7, verbose=1),
    callbacks.EarlyStopping(monitor='val_loss', patience=8, restore_best_weights=True)
]

print("--- Treinando Modelo Híbrido com Punições Fortes ---")
history = model.fit(
    train_ds, 
    validation_data=test_ds, 
    epochs=50, 
    class_weight=class_weight_dict,
    callbacks=callbacks_list
)

# Matriz de Confusão Final
y_pred = np.argmax(model.predict(test_ds), axis=1)
plt.figure(figsize=(10, 8))
sns.heatmap(confusion_matrix(y_ts, y_pred), annot=True, fmt='d', xticklabels=le.classes_, yticklabels=le.classes_, cmap='Blues')
plt.title("Matriz de Confusão Final (Focal Loss + Dynamic LR)")
plt.show()

print(classification_report(y_ts, y_pred, target_names=le.classes_))